# ImageSimilarity dev smoke test

Hyper-minimal Colab notebook for testing the new manifest-based flow on a small Drive-sourced subset:

1. Mount Drive and reuse the same default Drive paths as `run.ipynb`.
2. Unzip source/target image archives into local Colab storage.
3. Copy a small sampled subset into separate dev folders.
4. Run `match_new.py` to produce `retrieval_manifest.jsonl` + `feature_cache.npz`.
5. Run `aspanfilter.py` to produce ASpan-filtered manifests and sidecar `.npz` files.

Edit only the config cell for most tests.


In [ ]:
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
from pathlib import Path

# Same project root convention as run.ipynb.
DRIVE_ROOT = Path('/content/drive/MyDrive/ImageSimilarity')
LOCAL_ROOT = Path('/content/image_similarity_dev')

# Input archives from Drive. Change these two lines if you want to test a different import.
SOURCE_ZIP = DRIVE_ROOT / 'workingsourcecrops.zip'
TARGET_ZIP = DRIVE_ROOT / 'working_targetcrops.zip'

# Repo scripts / model files on Drive.
MATCH_NEW_SCRIPT = DRIVE_ROOT / 'match_new.py'
ASPAN_FILTER_SCRIPT = DRIVE_ROOT / 'aspanfilter.py'
MODEL = DRIVE_ROOT / 'ModelComboDINO.py'
MODEL_CLASS = 'ModelComboDINO'

# Checkpoints / ASpanFormer paths, matching run.ipynb defaults.
WEIGHTS_DIR = LOCAL_ROOT / 'weights'
MATCH_WEIGHTS = WEIGHTS_DIR / 'best.pt'
DINO_CHECKPOINT_DRIVE = DRIVE_ROOT / 'DINO.pt'
# ModelComboDINO expects this DINOv3 weight filename relative to the process cwd.
DINO_V3_CHECKPOINT_NAME = 'dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth'
DINO_V3_CHECKPOINT_DRIVE = DRIVE_ROOT / DINO_V3_CHECKPOINT_NAME
DINO_V3_CHECKPOINT_LOCAL = LOCAL_ROOT / DINO_V3_CHECKPOINT_NAME
ASPAN_ROOT = DRIVE_ROOT / 'ml-aspanformer'
ASPAN_CONFIG = ASPAN_ROOT / 'configs/aspan/outdoor/aspan_test.py'
ASPAN_WEIGHTS = DRIVE_ROOT / 'aspanweights/outdoor.ckpt'

# Local working dirs.
SOURCE_DIR = LOCAL_ROOT / 'source_all'
TARGET_DIR = LOCAL_ROOT / 'target_all'
SAMPLE_SOURCE_DIR = LOCAL_ROOT / 'source_sample'
SAMPLE_TARGET_DIR = LOCAL_ROOT / 'target_sample'
RETRIEVAL_OUTPUT_DIR = LOCAL_ROOT / 'retrieval_output'
ASPAN_OUTPUT_DIR = LOCAL_ROOT / 'aspan_output'

# Dev-size knobs. Increase only after the small smoke test works.
RESET_LOCAL_ROOT = True
SAMPLE_SOURCE_N = 5
SAMPLE_TARGET_N = 25
SAMPLE_SEED = 7
TOPX = 5
CHUNK_SIZE = 256
RUN_ASPAN_FILTERING = True
BREAKPOINT_VALUE = 50
LONG_DIM = 1024
ASPAN_MAX_PAIRS = 10  # set to None to process all retrieved candidates

# VGGT-Omega judging stage. This starts from aspanfilter.py outputs; it does not rerun ASpanFormer.
RUN_VGGT_JUDGE = True
VGGT_JUDGE_SCRIPT = DRIVE_ROOT / 'vggt_judge.py'
VGGT_OMEGA_CHECKPOINT_DRIVE = DRIVE_ROOT / 'weights/VGGT-Omega/vggt_omega_1b_512.pt'
VGGT_OMEGA_CHECKPOINT_LOCAL = LOCAL_ROOT / 'vggt_omega_1b_512.pt'
VGGT_OUTPUT_DIR = LOCAL_ROOT / 'vggt_output'
VGGT_MAX_RES = 384
VGGT_GLOBAL_SIM_THRESHOLD = 0.90
VGGT_POSE_SHIFT_THRESHOLD = 0.10
VGGT_MAX_PAIRS = None  # set to a small integer such as 5 for first smoke tests
# Resume only helps if the previous VGGT_OUTPUT_DIR still exists. If you rerun the
# staging/reset cell with RESET_LOCAL_ROOT=True, local VGGT outputs are deleted.
VGGT_RESUME = True

print('DRIVE_ROOT:', DRIVE_ROOT)
print('LOCAL_ROOT:', LOCAL_ROOT)
print('SOURCE_ZIP:', SOURCE_ZIP)
print('TARGET_ZIP:', TARGET_ZIP)
print('MATCH_WEIGHTS:', MATCH_WEIGHTS, '(copied from', DINO_CHECKPOINT_DRIVE, 'if present)')


In [ ]:
import json
import os
import random
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff'}


def run_command(cmd, cwd=None):
    printable = ' '.join(str(x) for x in cmd)
    print(f'\n$ {printable}')
    subprocess.run([str(x) for x in cmd], cwd=str(cwd) if cwd else None, check=True)


def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)
    return Path(path)


def copy_file(src, dst):
    src = Path(src)
    dst = Path(dst)
    if not src.exists():
        raise FileNotFoundError(src)
    ensure_dir(dst.parent)
    shutil.copy2(src, dst)
    print(f'Copied {src} -> {dst}')
    return dst


def unzip_archive(zip_path, output_dir):
    zip_path = Path(zip_path)
    output_dir = ensure_dir(output_dir)
    if not zip_path.exists():
        raise FileNotFoundError(zip_path)
    print(f'Unzipping {zip_path} -> {output_dir}')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(output_dir)
    return output_dir


def image_files(root):
    root = Path(root)
    return sorted(p for p in root.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def sample_image_tree(input_dir, output_dir, sample_n=None, seed=0):
    # Copy a random image-file subset while preserving relative paths.
    # Point this at any extracted Drive import directory to make a smaller
    # folder tree for quick matching/filtering tests.
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)
    files = image_files(input_dir)
    if not files:
        raise ValueError(f'No image files found under {input_dir}')

    if output_dir.exists():
        shutil.rmtree(output_dir)
    ensure_dir(output_dir)

    if sample_n is None or sample_n >= len(files):
        chosen = files
    else:
        rng = random.Random(seed)
        chosen = sorted(rng.sample(files, sample_n))

    for src in chosen:
        rel = src.relative_to(input_dir)
        dst = output_dir / rel
        ensure_dir(dst.parent)
        shutil.copy2(src, dst)

    print(f'Sampled {len(chosen)} / {len(files)} images from {input_dir} -> {output_dir}')
    return chosen


def list_tree_summary(path, max_items=20):
    path = Path(path)
    print(f'\nTree summary: {path}')
    if not path.exists():
        print('  [missing]')
        return
    files = sorted(p for p in path.rglob('*') if p.is_file())
    print(f'  files: {len(files)}')
    for p in files[:max_items]:
        print(' ', p.relative_to(path))
    if len(files) > max_items:
        print(f'  ... {len(files) - max_items} more')


In [ ]:
# Optional but useful in a fresh Colab runtime: install the packages used by the model/ASpanFormer path.
# If a package is already present this should be quick/no-op-ish.
run_command([sys.executable, '-m', 'pip', 'install', '-q', 'torchmetrics', 'albumentations', 'pytorch-lightning'])
if (ASPAN_ROOT / 'requirements.txt').exists():
    run_command([sys.executable, '-m', 'pip', 'install', '-q', '-r', ASPAN_ROOT / 'requirements.txt'])
else:
    print('No ASpanFormer requirements.txt found at', ASPAN_ROOT / 'requirements.txt')


In [ ]:
# Stage Drive imports into local Colab storage, then make a small source/target subset.
if RESET_LOCAL_ROOT and LOCAL_ROOT.exists():
    print('Resetting', LOCAL_ROOT)
    shutil.rmtree(LOCAL_ROOT)

for path in [LOCAL_ROOT, WEIGHTS_DIR, SOURCE_DIR, TARGET_DIR, RETRIEVAL_OUTPUT_DIR, ASPAN_OUTPUT_DIR]:
    ensure_dir(path)

copy_file(SOURCE_ZIP, LOCAL_ROOT / SOURCE_ZIP.name)
copy_file(TARGET_ZIP, LOCAL_ROOT / TARGET_ZIP.name)
unzip_archive(LOCAL_ROOT / SOURCE_ZIP.name, SOURCE_DIR)
unzip_archive(LOCAL_ROOT / TARGET_ZIP.name, TARGET_DIR)

if DINO_CHECKPOINT_DRIVE.exists():
    copy_file(DINO_CHECKPOINT_DRIVE, MATCH_WEIGHTS)
else:
    print('WARNING: no Drive checkpoint found at', DINO_CHECKPOINT_DRIVE)
    print('Set DINO_CHECKPOINT_DRIVE or MATCH_WEIGHTS in the config cell before running match_new.py.')

if DINO_V3_CHECKPOINT_DRIVE.exists():
    copy_file(DINO_V3_CHECKPOINT_DRIVE, DINO_V3_CHECKPOINT_LOCAL)
else:
    print('WARNING: no DINOv3 base checkpoint found at', DINO_V3_CHECKPOINT_DRIVE)
    print('ModelComboDINO may try to download or fail unless this file is available in the run cwd.')

sample_image_tree(SOURCE_DIR, SAMPLE_SOURCE_DIR, SAMPLE_SOURCE_N, SAMPLE_SEED)
sample_image_tree(TARGET_DIR, SAMPLE_TARGET_DIR, SAMPLE_TARGET_N, SAMPLE_SEED + 1)
list_tree_summary(SAMPLE_SOURCE_DIR)
list_tree_summary(SAMPLE_TARGET_DIR)


In [ ]:
# Run the new representation-retrieval step.
# Output: retrieval_manifest.jsonl + feature_cache.npz under RETRIEVAL_OUTPUT_DIR.
if not MATCH_NEW_SCRIPT.exists():
    raise FileNotFoundError(MATCH_NEW_SCRIPT)
if not MODEL.exists():
    raise FileNotFoundError(MODEL)
if not MATCH_WEIGHTS.exists():
    raise FileNotFoundError(f'Missing weights: {MATCH_WEIGHTS}')

run_command([
    sys.executable,
    MATCH_NEW_SCRIPT,
    '--weights', MATCH_WEIGHTS,
    '--model-definition', MODEL,
    '--model-class', MODEL_CLASS,
    '--source', SAMPLE_SOURCE_DIR,
    '--target', SAMPLE_TARGET_DIR,
    '--topx', TOPX,
    '--output-dir', RETRIEVAL_OUTPUT_DIR,
    '--chunk-size', CHUNK_SIZE,
], cwd=LOCAL_ROOT)

retrieval_manifest = RETRIEVAL_OUTPUT_DIR / 'retrieval_manifest.jsonl'
feature_cache = RETRIEVAL_OUTPUT_DIR / 'feature_cache.npz'
print('retrieval_manifest:', retrieval_manifest, retrieval_manifest.exists())
print('feature_cache:', feature_cache, feature_cache.exists())


In [ ]:
# Run the new ASpanFormer filter step on the retrieval manifest.
# Output: aspan_all_manifest.jsonl, vggt_candidates_manifest.jsonl, and sidecar NPZs.
if RUN_ASPAN_FILTERING:
    if not ASPAN_FILTER_SCRIPT.exists():
        raise FileNotFoundError(ASPAN_FILTER_SCRIPT)
    if not retrieval_manifest.exists():
        raise FileNotFoundError(retrieval_manifest)

    cmd = [
        sys.executable,
        ASPAN_FILTER_SCRIPT,
        '--input-manifest', retrieval_manifest,
        '--output-dir', ASPAN_OUTPUT_DIR,
        '--breakpoint-value', BREAKPOINT_VALUE,
        '--aspanpath', ASPAN_ROOT,
        '--weights_path', ASPAN_WEIGHTS,
        '--config_path', ASPAN_CONFIG,
        '--long_dim', LONG_DIM,
        '--resume',
        '--progress-every', 1,
    ]
    if ASPAN_MAX_PAIRS is not None:
        cmd += ['--max-pairs', ASPAN_MAX_PAIRS]
    run_command(cmd, cwd=LOCAL_ROOT)
else:
    print('RUN_ASPAN_FILTERING=False; skipping aspanfilter.py')


In [ ]:
# Run VGGT-Omega judge on ASpan-passed candidate pairs.
# This consumes aspanfilter.py output; it does not rerun ASpanFormer.
if RUN_VGGT_JUDGE:
    run_command([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/facebookresearch/vggt-omega.git'])

    ensure_dir(VGGT_OUTPUT_DIR)
    if not VGGT_JUDGE_SCRIPT.exists():
        raise FileNotFoundError(f'Missing VGGT judge script: {VGGT_JUDGE_SCRIPT}')
    if not VGGT_OMEGA_CHECKPOINT_DRIVE.exists():
        raise FileNotFoundError(f'Missing VGGT-Omega checkpoint: {VGGT_OMEGA_CHECKPOINT_DRIVE}')
    copy_file(VGGT_OMEGA_CHECKPOINT_DRIVE, VGGT_OMEGA_CHECKPOINT_LOCAL)

    vggt_input_manifest = ASPAN_OUTPUT_DIR / 'vggt_candidates_manifest.jsonl'
    if not vggt_input_manifest.exists():
        raise FileNotFoundError(f'Missing aspanfilter output manifest: {vggt_input_manifest}')

    cmd = [
        sys.executable,
        VGGT_JUDGE_SCRIPT,
        '--input-manifest', vggt_input_manifest,
        '--output-dir', VGGT_OUTPUT_DIR,
        '--checkpoint', VGGT_OMEGA_CHECKPOINT_LOCAL,
        '--global-sim-threshold', str(VGGT_GLOBAL_SIM_THRESHOLD),
        '--pose-shift-threshold', str(VGGT_POSE_SHIFT_THRESHOLD),
        '--max-res', str(VGGT_MAX_RES),
        '--progress-every', 1,
    ]
    if VGGT_RESUME:
        cmd.append('--resume')
    if VGGT_MAX_PAIRS is not None:
        cmd += ['--max-pairs', str(VGGT_MAX_PAIRS)]

    run_command(cmd, cwd=LOCAL_ROOT)

    for path in [
        VGGT_OUTPUT_DIR / 'vggt_judged_manifest.jsonl',
        VGGT_OUTPUT_DIR / 'true_matches_manifest.jsonl',
        VGGT_OUTPUT_DIR / 'vggt_judge_summary.json',
    ]:
        print(path, 'exists=', path.exists())
        if path.suffix == '.jsonl' and path.exists():
            print('rows=', count_jsonl(path) if 'count_jsonl' in globals() else sum(1 for line in path.open() if line.strip()))
else:
    print('RUN_VGGT_JUDGE=False; skipping VGGT-Omega judge')


In [ ]:
# Inspect the small smoke-test outputs.
def count_jsonl(path):
    path = Path(path)
    if not path.exists():
        return 0
    with path.open() as f:
        return sum(1 for line in f if line.strip())

outputs = {
    'retrieval_manifest': RETRIEVAL_OUTPUT_DIR / 'retrieval_manifest.jsonl',
    'feature_cache': RETRIEVAL_OUTPUT_DIR / 'feature_cache.npz',
    'aspan_all_manifest': ASPAN_OUTPUT_DIR / 'aspan_all_manifest.jsonl',
    'vggt_candidates_manifest': ASPAN_OUTPUT_DIR / 'vggt_candidates_manifest.jsonl',
    'vggt_judged_manifest': VGGT_OUTPUT_DIR / 'vggt_judged_manifest.jsonl',
    'true_matches_manifest': VGGT_OUTPUT_DIR / 'true_matches_manifest.jsonl',
    'vggt_judge_summary': VGGT_OUTPUT_DIR / 'vggt_judge_summary.json',
}
for name, path in outputs.items():
    rows = count_jsonl(path) if path.suffix == '.jsonl' else 'n/a'
    print(f'{name}: {path} exists={path.exists()} rows={rows}')

for name in ['retrieval_manifest', 'aspan_all_manifest', 'vggt_candidates_manifest']:
    path = outputs[name]
    if path.exists():
        print(f'\nFirst row from {name}:')
        with path.open() as f:
            for line in f:
                if line.strip():
                    print(json.dumps(json.loads(line), indent=2)[:2000])
                    break

list_tree_summary(ASPAN_OUTPUT_DIR, max_items=30)
